### Check Python version

In [1]:
from platform import python_version_tuple

assert (
    python_version_tuple()[:2] == ("3", "11")
), "You are not running Python 3.11. Make sure to run Python through the course Conda environment."

In [2]:
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm import tqdm
import tifffile as tiff

from from_rgb_labels_to_labels import from_rgb_labels_to_int_labels


# EXTRACTION OF IMAGES STARTS HERE

- from each original image tile (6000 x 6000 x 3)
    - we will extract patches of size (512 x 512 x 3)
    - `stride = 256` therefore, the neighboring patches will overlap by half

- To be able to use all pixels from the original image tiles
    - A last index corresponding to 6000-512 will be manually added
    - in order to cover the rightmost part and the bottom part of each original image tile

In [3]:
# =====================================================
# CONFIGURATION
# =====================================================

IMAGE_DIR = Path("potsdam/Ortho_RGB_train")
LABEL_DIR = Path("potsdam/Labels_all")

OUT_IMAGE_DIR = Path("train_patches/images")
OUT_LABEL_DIR = Path("train_patches/labels")

PATCH_SIZE = 512
STRIDE = 256  # 512 = no overlap

OUT_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
OUT_LABEL_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
# =====================================================
# EXTRACTION of Train patches
# =====================================================

image_files = sorted(IMAGE_DIR.glob("*.tif"))


for image_path in tqdm(image_files):

    # Labels have same name as image with ending "_RGB" replaced with "_label")
    label_path = LABEL_DIR / image_path.name.replace("_RGB", "_label")

    if not label_path.exists():
        print(f"Missing label for {image_path.name}")
        continue

    image = tiff.imread(image_path)
    label = tiff.imread(label_path)

    # shape of image read using tifffile is (H, W, C) 
    # for Potsdam images, this is== (6000, 6000, 3) 
    h, w = image.shape[0], image.shape[1]


    stem = (image_path.stem).replace("_RGB", "")

    patch_count = 0
    skip_count = 0

    # Calculate the x and y starting coordinates of the patches
    xs = list(range(0, w - PATCH_SIZE, STRIDE))
    ys = list(range(0, h - PATCH_SIZE, STRIDE))

    # Ensure that the last patch along x-axis includes the rightmost part of the image
    if xs[-1] != w - PATCH_SIZE:
        xs.append(w - PATCH_SIZE)

    # Ensure that the last patch along y-axis includes the bottom edge of the image
    if ys[-1] != h - PATCH_SIZE:
        ys.append(h - PATCH_SIZE)

    
    for y in ys:
        for x in xs:

            image_patch = image[
                y:y + PATCH_SIZE,
                x:x + PATCH_SIZE ]

            label_patch = label[
                y:y + PATCH_SIZE,
                x:x + PATCH_SIZE ]


            # create a new label patch with integer labels {0,1,2,3,4,5} instead of RGB values
            label_patch_int = from_rgb_labels_to_int_labels(label_patch)

            # Calculate the fraction of the patch that belongs to the majority class
            majority_fraction = np.max(np.bincount(label_patch_int.flatten())) / label_patch_int.size

            # If the majority class occupies more than 95% of the patch, skip this patch
            # i.e. if the patch is highly uniform, we skip it
            if majority_fraction > 0.95:
                skip_count += 1
                continue

            
            patch_name = f"{stem}_{y}_{x}.tif"
            patch_label_name = f"{stem}_{y}_{x}_label.tif"

            tiff.imwrite(
                OUT_IMAGE_DIR / patch_name,
                image_patch )

            tiff.imwrite(
                OUT_LABEL_DIR / patch_label_name,
                label_patch )

            patch_count += 1

    print(f"From image tile {image_path.name}: {patch_count} patches extracted, {skip_count} patches skipped due to majority class > 95%")

print("Finished.")

  4%|▍         | 1/24 [00:18<06:58, 18.19s/it]

From image tile top_potsdam_2_10_RGB.tif: 505 patches extracted, 24 patches skipped due to majority class > 95%


  8%|▊         | 2/24 [00:37<06:49, 18.62s/it]

From image tile top_potsdam_2_11_RGB.tif: 499 patches extracted, 30 patches skipped due to majority class > 95%


 12%|█▎        | 3/24 [00:55<06:27, 18.43s/it]

...

From image tile top_potsdam_7_7_RGB.tif: 528 patches extracted, 1 patches skipped due to majority class > 95%


 96%|█████████▌| 23/24 [08:10<00:23, 23.13s/it]

From image tile top_potsdam_7_8_RGB.tif: 528 patches extracted, 1 patches skipped due to majority class > 95%


100%|██████████| 24/24 [08:32<00:00, 21.37s/it]

From image tile top_potsdam_7_9_RGB.tif: 492 patches extracted, 37 patches skipped due to majority class > 95%
Finished.
